# 01 - Build the household table (CIS 2018-2022)

DRAFT code written by Claude (AI assistant) for study and verification. Fill in the Purpose / What the code does / My result fields in your own words after you have run and understood each cell.

Input: the five `CIS{year}_PUMF.txt` files (one row per person). Output: one row per household (the major income earner's row), one-family households only, five years stacked, saved as a CSV together with three small summary tables (missing values, sample flow, income quintile cut-offs).

### Step 1 - Libraries and folders

**Purpose:** 

**What the code does:** 

**Learn from:** [os.makedirs](https://docs.python.org/3/library/os.html#os.makedirs), [pandas.read_fwf](https://pandas.pydata.org/docs/reference/api/pandas.read_fwf.html)

**My result:** 

In [ ]:
# ================= SETTINGS: the only lines to change when reproducing this work =================
# 1. Download the five Canadian Income Survey public-use files (CIS2018_PUMF.txt ... CIS2022_PUMF.txt) from Statistics Canada
#    and put them in a folder called ProjectData inside base_path. They are large files, so they are not stored on GitHub.
# 2. Google Colab with Google Drive: keep the path below, or change it to wherever you put the folder.
#    Your own computer (no Drive): replace it with a local folder, e.g. './CIND820_CapstoneProject'.
# 3. Run notebook 01, then 02. The results are written to ProcessedData and EDA_outputs inside base_path.
base_path = '/content/drive/MyDrive/CIND820_CapstoneProject'
# ===================================================================================================

# --- Libraries ---
import os                  # file paths and folders (Python standard library)
import numpy as np         # numbers; used here for the weighted average, square root and weighted quintiles
import pandas as pd        # tables (DataFrames) and the fixed-width file reader pd.read_fwf

# --- Google Drive ---
# In Colab this connects your Drive. The try/except lets the same cell run outside Colab without an error.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# --- Folders (built from base_path) ---
folder_path = f'{base_path}/ProjectData'      # the five CIS{year}_PUMF.txt files
output_path = f'{base_path}/ProcessedData'    # the household table will be saved here
os.makedirs(output_path, exist_ok=True)   # create the output folder if it does not exist (no error if it does)

years = [2018, 2019, 2020, 2021, 2022]

### Step 2 - Which variables to read, and where they sit in each file

**Purpose:** 

**What the code does:** 

**Learn from:** The 'Table of Contents' of each year's data dictionary (Position column); [pandas.read_fwf](https://pandas.pydata.org/docs/reference/api/pandas.read_fwf.html)

**My result:** 

In [ ]:
# --- Where each variable sits in each year's file ---
# Each pair is (start, end) for pandas.read_fwf. The data dictionary counts positions from 1 and both ends are
# included, e.g. FSCHHLDM 1178-1178. read_fwf counts from 0 and excludes the end, so (start - 1, end).
# These numbers were read from the Table of Contents of each year's dictionary (2021 and 2022 are shifted
# because those files have more variables, and FWEIGHT and the dollar variables are wider).
pos = {
    2018: {
        'YEAR': (0, 4), 'PUMFID': (4, 9), 'FWEIGHT': (29, 39), 'HHMJIE': (1119, 1120),
        'FSCHHLDM': (1177, 1178), 'HHCOMP': (1118, 1119), 'HHSIZE': (1116, 1118), 'PROV': (39, 41),
        'DWTENR': (1145, 1146), 'MORTG': (1148, 1149), 'HLEV2G': (51, 52), 'IMMST': (77, 78),
        'MAJRI': (223, 225), 'EFATINC': (450, 458), 'ALFST': (57, 59), 'SEX': (47, 48),
        'AGEGP': (45, 47), 'EFAGYFMP': (430, 432),
    },
    2019: {
        'YEAR': (0, 4), 'PUMFID': (4, 9), 'FWEIGHT': (29, 39), 'HHMJIE': (1119, 1120),
        'FSCHHLDM': (1177, 1178), 'HHCOMP': (1118, 1119), 'HHSIZE': (1116, 1118), 'PROV': (39, 41),
        'DWTENR': (1145, 1146), 'MORTG': (1148, 1149), 'HLEV2G': (51, 52), 'IMMST': (77, 78),
        'MAJRI': (223, 225), 'EFATINC': (450, 458), 'ALFST': (57, 59), 'SEX': (47, 48),
        'AGEGP': (45, 47), 'EFAGYFMP': (430, 432),
    },
    2020: {
        'YEAR': (0, 4), 'PUMFID': (4, 9), 'FWEIGHT': (29, 39), 'HHMJIE': (1119, 1120),
        'FSCHHLDM': (1177, 1178), 'HHCOMP': (1118, 1119), 'HHSIZE': (1116, 1118), 'PROV': (39, 41),
        'DWTENR': (1145, 1146), 'MORTG': (1148, 1149), 'HLEV2G': (51, 52), 'IMMST': (77, 78),
        'MAJRI': (231, 233), 'EFATINC': (450, 458), 'ALFST': (57, 59), 'SEX': (47, 48),
        'AGEGP': (45, 47), 'EFAGYFMP': (430, 432),
    },
    2021: {
        'YEAR': (0, 4), 'PUMFID': (4, 9), 'FWEIGHT': (29, 41), 'HHMJIE': (1784, 1785),
        'FSCHHLDM': (1842, 1843), 'HHCOMP': (1783, 1784), 'HHSIZE': (1781, 1783), 'PROV': (41, 43),
        'DWTENR': (1810, 1811), 'MORTG': (1813, 1814), 'HLEV2G': (53, 54), 'IMMST': (78, 79),
        'MAJRI': (356, 358), 'EFATINC': (679, 691), 'ALFST': (59, 61), 'SEX': (49, 50),
        'AGEGP': (47, 49), 'EFAGYFMP': (651, 653),
    },
    2022: {
        'YEAR': (0, 4), 'PUMFID': (4, 9), 'FWEIGHT': (29, 41), 'HHMJIE': (1820, 1821),
        'FSCHHLDM': (1878, 1879), 'HHCOMP': (1819, 1820), 'HHSIZE': (1817, 1819), 'PROV': (41, 43),
        'DWTENR': (1846, 1847), 'MORTG': (1849, 1850), 'HLEV2G': (53, 54), 'IMMST': (78, 79),
        'MAJRI': (368, 370), 'EFATINC': (691, 703), 'ALFST': (59, 61), 'SEX': (49, 50),
        'AGEGP': (47, 49), 'EFAGYFMP': (663, 665),
    },
}



# --- Variables used in the analysis ---
# The nine predictors of the model, as they are named in the household table. Three are built in this notebook:
#   income_quintile - from EFATINC (after-tax income of the economic family) divided by the square root of HHSIZE,
#                     then split into weighted fifths WITHIN each year (no price adjustment across years)
#   tenure          - from DWTENR (dwelling owned or not) and MORTG (mortgage or not)
#   has_children    - from EFAGYFMP (age group of the youngest person in the economic family):
#                     1 = at least one person under 18 (codes 01 to 04), 0 = adults only
#   edu_household   - from HLEV2G: the HIGHEST education among the adults (18+) in the household, the way most
#                     food-insecurity studies define it. Missing only if no adult gave a valid answer.
#                     To use the MAJOR INCOME EARNER's own education instead, replace 'edu_household' with 'HLEV2G'
#                     in the list below (and in notebook 02). Both columns are saved in the household table either way.
# The other five are read as they are. SEX, IMMST and MAJRI belong to the major income earner (the household's row
# we keep).
predictors = ['income_quintile', 'tenure', 'has_children', 'SEX', 'edu_household', 'IMMST', 'MAJRI', 'PROV', 'HHSIZE']

# Variables whose missing values are counted for the missingness table (the ones that feed the predictors).
# ALFST is NOT a predictor (dropped: about 17-21% of earners are a valid skip, a structural gap). It is read only so
# that its missing-value counts stay in the table as the evidence for that decision.
# DWTENR and MORTG are not listed: they only build `tenure`, and MORTG is a valid skip by design for households that
# do not own their dwelling (that is the 'not owned' group), so its 'missing' share would be misleading.
source_vars = ['PROV', 'HHSIZE', 'SEX', 'EFAGYFMP', 'HLEV2G', 'edu_household', 'IMMST', 'MAJRI', 'EFATINC', 'ALFST']
# (For edu_household the 'blank' column of the missingness table means: no adult in the household gave a valid answer.)

# --- Special codes ---
# Answer categories from the dictionaries: (valid skip code, not stated code); None = the dictionary has no such code.
# They are not real answers and are turned into missing values below. Variables not listed have no special codes.
special_codes = {
    'HLEV2G': (6, 9),
    'IMMST':  (6, 9),
    'MAJRI':  (96, None),   # valid skip = under 15/16 years old, so no earner should have it
    'MORTG':  (6, None),    # valid skip = the dwelling is not owned (the question applies to owners only)
    'ALFST':  (96, 99),
}

# --- Numbers to check my results against ---
# Person records and weighted persons: 'YEAR' variable, Total row, in each year's dictionary (page 7).
expected_persons = {2018: 94336, 2019: 72643, 2020: 123718, 2021: 89939, 2022: 96302}
expected_weighted_persons = {2018: 36329666, 2019: 36831173, 2020: 37101086, 2021: 37464631, 2022: 38007991}
# Households (all of them, before the one-family restriction): your own count from notebook 00, Step 4.
expected_households = {2018: 40857, 2019: 31658, 2020: 53882, 2021: 39243, 2022: 41978}
# Weighted % of PERSONS living in a food-insecure household = (weighted frequency of codes 1+2+3) / (weighted total),
# both printed in each year's FSCHHLDM entry (dictionary pages 79, 79, 79, 86, 87).
expected_pct_food_insecure = {2018: 16.38, 2019: 15.85, 2020: 15.76, 2021: 18.56, 2022: 22.85}

### Step 3 - Two small helpers: a checker and a weighted-quintile function

**Purpose:** 

**What the code does:** 

**Learn from:** [Python f-strings](https://docs.python.org/3/tutorial/inputoutput.html#formatted-string-literals), [numpy.cumsum](https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html), [numpy.searchsorted](https://numpy.org/doc/stable/reference/generated/numpy.searchsorted.html)

**My result:** 

In [ ]:
# A helper that prints PASS or FAIL for each check. It does not stop the notebook, so you can see every result.
def check(label, ok):
    print(('PASS  ' if ok else 'FAIL  ') + label)
    return bool(ok)


def weighted_quintile(values, weights):
    """Split households into five income groups of equal WEIGHTED size (1 = lowest fifth, 5 = highest fifth).
    values  : a pandas Series of incomes
    weights : a pandas Series of survey weights (same index)
    Households with exactly the same income always land in the same group (ties are not split).
    Returns the group of each household and the four cut-off values."""
    v = values.to_numpy(dtype=float)
    w = weights.to_numpy(dtype=float)
    order = np.argsort(v, kind='stable')                    # positions that sort the incomes from low to high
    cum_share = np.cumsum(w[order]) / w.sum()               # share of all weight at or below each household
    # Cut-off for each step = the first sorted income where the cumulative share reaches 20%, 40%, 60%, 80%
    cutoffs = [v[order][np.searchsorted(cum_share, p)] for p in (0.2, 0.4, 0.6, 0.8)]
    # Group = 1 + the number of cut-offs that are strictly below the household's income
    group = 1 + sum((v > c).astype(int) for c in cutoffs)
    return pd.Series(group, index=values.index), cutoffs

### Step 4 - Read each year, check it, keep one row per household, build the predictors

**Purpose:** 

**What the code does:** 

**Learn from:** [pandas.read_fwf](https://pandas.pydata.org/docs/reference/api/pandas.read_fwf.html), [DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html), [numpy.average](https://numpy.org/doc/stable/reference/generated/numpy.average.html), [numpy.select](https://numpy.org/doc/stable/reference/generated/numpy.select.html), [numpy.sqrt](https://numpy.org/doc/stable/reference/generated/numpy.sqrt.html), [Series.mask](https://pandas.pydata.org/docs/reference/api/pandas.Series.mask.html), [nullable integers](https://pandas.pydata.org/docs/user_guide/integer_na.html)

**My result:** 

In [ ]:
household_frames = []   # one household table per year; stacked after the loop
missing_rows = []       # one row per variable per year, for the missingness table
flow_rows = []          # one row per year: how many households were dropped by the one-family restriction
cutoff_rows = []        # one row per year: income quintile cut-offs (dollars) and the weighted share in each quintile

for year in years:
    print(f'\n===== {year} =====')

    # 1. Read the fixed-width file: only the columns listed in pos[year]; every column read as a number.
    cols = pos[year]
    persons = pd.read_fwf(
        f'{folder_path}/CIS{year}_PUMF.txt',
        colspecs=list(cols.values()),   # (start, end) pairs
        names=list(cols.keys()),        # column names
        dtype='float64',                # read everything as numbers; blanks (if any) become NaN instead of an error
    )   # a dollar value such as -0183100 is read as the number -183100

    # 2. Checks on the PERSON-level data against numbers printed in the dictionary
    check(f'{year}: person rows = dictionary total ({expected_persons[year]:,})', len(persons) == expected_persons[year])
    check(f'{year}: YEAR column holds only {year}', (persons['YEAR'] == year).all())
    blanks = persons.isna().sum()
    print('      blank cells per column (columns with blanks only):', blanks[blanks > 0].to_dict())

    weighted_total = persons['FWEIGHT'].sum()
    ratio = weighted_total / expected_weighted_persons[year]
    check(f'{year}: sum of FWEIGHT is within 0.05% of the dictionary total (sum/dictionary = {ratio:.6f})',
          abs(ratio - 1) < 0.0005)
    # If this FAILs with a ratio near 10,000 or 0.0001, FWEIGHT has implied decimals: divide it accordingly.

    pct_fi = 100 * np.average(persons['FSCHHLDM'] > 0, weights=persons['FWEIGHT'])
    check(f'{year}: weighted % of persons in food-insecure households = {pct_fi:.2f} (dictionary {expected_pct_food_insecure[year]})',
          abs(pct_fi - expected_pct_food_insecure[year]) <= 0.05)
    check(f'{year}: FSCHHLDM only has codes 0 to 3', persons['FSCHHLDM'].isin([0, 1, 2, 3]).all())
    check(f'{year}: HHCOMP only has codes 1 to 3', persons['HHCOMP'].isin([1, 2, 3]).all())
    check(f'{year}: AGEGP and EFAGYFMP only have codes 1 to 15', persons['AGEGP'].between(1, 15).all() and persons['EFAGYFMP'].between(1, 15).all())

    # 3. Household structure (same tests as notebook 00, but strict)
    earners = (persons['HHMJIE'] == 1).groupby(persons['PUMFID']).sum()      # major earners per household
    check(f'{year}: every household has exactly one major income earner', (earners == 1).all())
    check(f'{year}: household count = {expected_households[year]:,}', persons['PUMFID'].nunique() == expected_households[year])
    check(f'{year}: FSCHHLDM is identical within each household', persons.groupby('PUMFID')['FSCHHLDM'].nunique().max() == 1)

    # The economic-family income (EFATINC) is repeated on every member's row. It equals the HOUSEHOLD's income only
    # when the household is one economic family (HHCOMP 1 or 2). This check is the reason for the restriction below.
    one_family = persons['HHCOMP'].isin([1, 2])
    ef_values = persons[one_family].groupby('PUMFID')['EFATINC'].nunique()    # distinct EFATINC values per household
    check(f'{year}: EFATINC is the same for every member of a one-family household', (ef_values == 1).all())

    # Information only (not a pass/fail check): is HHSIZE the number of person rows? If 07 means "7 or more",
    # large households would differ.
    rows_per_household = persons.groupby('PUMFID').size()
    hhsize_per_household = persons.groupby('PUMFID')['HHSIZE'].first()
    print(f'      info: HHSIZE equals the number of person rows in '
          f'{100 * (rows_per_household == hhsize_per_household).mean():.1f}% of households')

    # Any member under 18? Age groups 01 to 04 are under 18 in every year: 0-5, 6-9, 10-15, 16-17 in 2018-2021 and
    # 0-5, 6-9, 10-14, 15-17 in 2022 (the cut between groups 03 and 04 moves, but together they always end at 17).
    member_under_18 = (persons['AGEGP'] <= 4).groupby(persons['PUMFID']).any()   # True if any household member is under 18

    # Highest education among the ADULTS (AGEGP 05 = 18 and over) with a real answer (HLEV2G codes 1 to 4; 6 = valid skip
    # and 9 = not stated are left out). Codes rise with education, so the maximum is the highest level.
    # Adults only, because the question starts at age 16 in 2018-2021 and at 15 in 2022, and teenagers still in school
    # should not raise the household's level.
    adult_education = persons['HLEV2G'].where((persons['AGEGP'] >= 5) & persons['HLEV2G'].between(1, 4))
    education_household_highest = adult_education.groupby(persons['PUMFID']).max()    # NaN if no adult has a valid answer

    # 4. Keep the major income earner's row = one row per household
    hh = persons[persons['HHMJIE'] == 1].copy()
    check(f'{year}: rows after filter = number of households', len(hh) == persons['PUMFID'].nunique())

    # 5. Keep one-family households only (HHCOMP 1 = one person, 2 = two or more persons in one economic family).
    #    HHCOMP 3 (two or more economic families) is dropped because EFATINC would then be only one family's income.
    n_all = len(hh)
    hh = hh[hh['HHCOMP'].isin([1, 2])].copy()
    print(f'      one-family households kept: {len(hh):,} of {n_all:,} (dropped {n_all - len(hh):,}, '
          f'{100 * (n_all - len(hh)) / n_all:.1f}%)')

    # 5a. Highest education in the household next to the earner's own (HLEV2G still holds its raw codes here)
    hh['edu_household'] = hh['PUMFID'].map(education_household_highest)
    both_valid = hh['edu_household'].notna() & hh['HLEV2G'].between(1, 4)
    print(f'      info: highest education in the household equals the earner\'s in '
          f'{100 * (hh.loc[both_valid, "edu_household"] == hh.loc[both_valid, "HLEV2G"]).mean():.1f}% of households '
          f'where both are known')

    # 5b. Children in the household: EFAGYFMP = age group of the youngest person in the economic family.
    #     In a one-family household (what we kept) that is the youngest person in the household.
    #     0 = adults only (youngest person is 18 or older), 1 = at least one person under 18.
    #     Built from ages, NOT from FSCCHLDM (child food security status), which would leak the outcome.
    hh['has_children'] = (hh['EFAGYFMP'] <= 4).astype(int)
    check(f'{year}: has_children from EFAGYFMP agrees with the members\' own age groups (AGEGP) in every household',
          (hh['PUMFID'].map(member_under_18).astype(int) == hh['has_children']).all())
    print(f'      households with at least one person under 18: {100 * hh["has_children"].mean():.1f}%')

    # 6. Count special codes BEFORE they are turned into missing values (valid skip and not stated are different things).
    #    Counted in the households that are kept.
    for var in source_vars:
        skip_code, notstated_code = special_codes.get(var, (None, None))
        skip_n = int((hh[var] == skip_code).sum()) if skip_code is not None else 0
        notstated_n = int((hh[var] == notstated_code).sum()) if notstated_code is not None else 0
        missing_rows.append({'YEAR': year, 'variable': var, 'households': len(hh),
                             'valid_skip': skip_n, 'not_stated': notstated_n, 'blank': int(hh[var].isna().sum())})

    # 7. Tenure in three groups, built from the raw codes (before step 8):
    #    DWTENR 1 = owned by a member of the household, 2 = not owned; MORTG 1 = mortgage, 2 = no mortgage, 6 = valid skip.
    #    1 = not owned, 2 = owned with a mortgage, 3 = owned without a mortgage. Anything else would be NaN.
    hh['tenure'] = np.select(
        [hh['DWTENR'] == 2,
         (hh['DWTENR'] == 1) & (hh['MORTG'] == 1),
         (hh['DWTENR'] == 1) & (hh['MORTG'] == 2)],
        [1, 2, 3],
        default=np.nan)
    check(f'{year}: every household got a tenure group', hh['tenure'].notna().all())
    check(f'{year}: MORTG is a valid skip (6) for every household that does not own its dwelling',
          (hh.loc[hh['DWTENR'] == 2, 'MORTG'] == 6).all())

    # 8. Special codes -> missing values (NaN)
    for var, codes in special_codes.items():
        hh[var] = hh[var].mask(hh[var].isin([c for c in codes if c is not None]))

    # 9. Household income adjusted for household size: divide by the square root of the number of people.
    #    Dollars of that year (no price adjustment). Zero and negative incomes are kept.
    check(f'{year}: HHSIZE is between 1 and 7 in every household', hh['HHSIZE'].between(1, 7).all())
    hh['adj_income'] = hh['EFATINC'] / np.sqrt(hh['HHSIZE'])

    # 10. Income quintile within this year, weighted by the earner's FWEIGHT (the household weight).
    #     No target variable is used here, only income and weights.
    hh['income_quintile'], cuts = weighted_quintile(hh['adj_income'], hh['FWEIGHT'])
    q_share = 100 * hh.groupby('income_quintile')['FWEIGHT'].sum() / hh['FWEIGHT'].sum()
    print('      weighted % of households in income quintiles 1 to 5:', q_share.round(1).tolist())
    cutoff_rows.append({'YEAR': year, 'cut_20': cuts[0], 'cut_40': cuts[1], 'cut_60': cuts[2], 'cut_80': cuts[3],
                        **{f'q{k}_weighted_pct': round(q_share[k], 2) for k in range(1, 6)}})

    flow_rows.append({'YEAR': year, 'households_all': n_all,
                      'dropped_two_or_more_economic_families': n_all - len(hh),
                      'households_analysis': len(hh),
                      'pct_dropped': round(100 * (n_all - len(hh)) / n_all, 2),
                      'adj_income_zero_or_negative': int((hh['adj_income'] <= 0).sum())})

    # 11. Whole-number columns become nullable integers (Int64) so that missing values can sit next to integers
    for c in hh.columns:
        if c not in ('FWEIGHT', 'EFATINC', 'adj_income'):
            hh[c] = hh[c].astype('Int64')

    # 12. Target: 0 = food secure, 1 = any food insecurity (marginal, moderate or severe)
    hh['food_insecure'] = (hh['FSCHHLDM'] > 0).astype(int)

    keep_cols = (['YEAR', 'PUMFID', 'FWEIGHT', 'FSCHHLDM', 'food_insecure'] + predictors +
                 ['HLEV2G', 'HHCOMP', 'DWTENR', 'MORTG', 'EFATINC', 'adj_income', 'EFAGYFMP'])   # predictors, target, supporting columns
    keep_cols = list(dict.fromkeys(keep_cols))   # drop repeats (HLEV2G is listed twice if it is also a predictor), keep the order
    household_frames.append(hh[keep_cols])

# --- Stack the five years ---
household_table = pd.concat(household_frames, ignore_index=True)   # pandas.concat puts the tables on top of each other
sample_flow = pd.DataFrame(flow_rows)
print('\nStacked table:', household_table.shape, '(households x columns); expected rows:',
      int(sample_flow['households_analysis'].sum()))
check('YEAR + PUMFID is a unique key', household_table.duplicated(['YEAR', 'PUMFID']).sum() == 0)
check('rows in the stacked table = households kept in the five years', len(household_table) == sample_flow['households_analysis'].sum())

# --- Save ---
household_table.to_csv(f'{output_path}/household_table.csv', index=False)
missingness = pd.DataFrame(missing_rows)
for c in ['valid_skip', 'not_stated', 'blank']:
    missingness[c + '_pct'] = (100 * missingness[c] / missingness['households']).round(2)
missingness.to_csv(f'{output_path}/missingness_by_year.csv', index=False)
sample_flow.to_csv(f'{output_path}/sample_flow_by_year.csv', index=False)
income_cutoffs = pd.DataFrame(cutoff_rows)
income_cutoffs.to_csv(f'{output_path}/income_quintile_cutoffs_by_year.csv', index=False)
print('Saved household_table.csv, missingness_by_year.csv, sample_flow_by_year.csv and income_quintile_cutoffs_by_year.csv to', output_path)
household_table.head()

### Step 5 - Look at what was built

**Purpose:** 

**What the code does:** 

**Learn from:** [DataFrame.info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html)

**My result:** 

In [ ]:
# Column types, non-missing counts, and the first rows of the stacked table
household_table.info()
# Households dropped by the one-family restriction, and households with zero or negative adjusted income
display(sample_flow)
# Income quintile cut-offs (adjusted income, dollars of each year) and the weighted % of households in each quintile
display(income_cutoffs.round(0))
# Missing values by variable and year, as % of the kept households (valid skip + not stated + blank)
display(missingness.pivot(index='variable', columns='YEAR', values='valid_skip_pct').add(
        missingness.pivot(index='variable', columns='YEAR', values='not_stated_pct')).add(
        missingness.pivot(index='variable', columns='YEAR', values='blank_pct')).round(2))